# Comparison with IMERG: regridding and significance

Comparison of ERA5, CRCM6-ERA5 (UBB) and IMERG over 2001–2014 (Sect. 3.1.2): the merged contributions
(`ALL/CONTRIB/contribution_ETCs_1000hPa_2001-2014_noTC.pkl`, precipitation, from `concat_contribution_ETCs.py
--period 2001-2014`) and the percentiles of CRCM6-ERA5 and IMERG are regridded onto the ERA5 grid; the significance
of the differences with IMERG is tested on the same regridded fields. The figures are made by
`PLOT/plot_map_IMERG_evaluation.ipynb`; both notebooks are run by `JOBS/run_IMERG_evaluation.sh`.

Methods of `regrid_over_ERA5_grid.ipynb`: conservative for the totals (`total_*`), nearest neighbour for the counts
(`count_*`), bilinear for the percentiles. Differences with that notebook:
- the longitudes of all the grids are put in 0..360, where the domain is contiguous (in -180..180 the ERA5 and
  IMERG grids are cut at the date line, and the cells on each side of the cut get wrong bounds);
- the cell bounds of the lat-lon grids (ERA5, IMERG) are the midpoints between centres (regular grids);
- the points outside the CRCM6 domain are set to NaN before regridding (they are 0 in the merged contributions),
  and the NaN are skipped: a target cell gets the mean of its valid part if that part covers at least half of it,
  NaN otherwise.

Significance of the differences with IMERG of EEcum (cumulative exceedance above the 99th percentile within
1000 km of a cyclone, total - p99 x count), as in `significance_masks.py`: Welch's test on the 14 yearly (or
seasonal) values, false discovery rate controlled at 0.10 over each map.

Outputs:
- `ALL/CONTRIB/contribution_ETCs_1000hPa_2001-2014_noTC_regridded_on_ERA5.pkl`: the three datasets on the ERA5 grid
  (same keys as the merged file), NaN outside the domain;
- `<SIM>/PR_Percentile/pr_<sim>_percentile_2001-2014_regridded_on_ERA5.nc` for UBB and IMERG;
- `ALL/SIGNIFICANCE/significance_vs_IMERG_2001-2014_on_ERA5_noTC.nc`: variable `EEcum_pr`, 1 where the difference
  with IMERG is significant, dims sim (ERA5, UBB) x season (ALL, DJF, MAM, JJA, SON) x grid.

In [1]:
import os
import pickle
import sys
import warnings
import numpy as np
import xarray as xr
import xesmf as xe

sys.path.insert(0, '/home/vdemeyer/TRACKING/KATJA/POSTPROCESSING')
from significance_masks import test_maps, with_annual, ALPHA_FDR

warnings.filterwarnings('ignore', category=UserWarning, module='xesmf')

PERIOD = '2001-2014'
TAG = '_noTC'
QUANTILE = 0.99
NA_THRES = 0.5   # a target cell is NaN if more than half of it is NaN or outside the source grid
BASE = '/home/vdemeyer/projects/rrg-gachon/vdemeyer'
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']
SIMS = ['ERA5', 'UBB', 'IMERG']
LATLON = ['ERA5', 'IMERG']
TESTED = ['ERA5', 'UBB']                 # datasets tested against IMERG
DIMS = ('latitude', 'longitude')         # ERA5 grid

/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/MPI/gcc12/openmpi4/esmf/8.8.0/lib/python3.11/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


## Inputs

In [2]:
with open(f'{BASE}/ALL/CONTRIB/contribution_ETCs_1000hPa_{PERIOD}{TAG}.pkl', 'rb') as f:
    merged = pickle.load(f)


def native_percentile(sim):
    with xr.open_dataset(f'{BASE}/{sim}/PR_Percentile/pr_{sim.lower()}_percentile_{PERIOD}.nc') as ds:
        return ds['pr'].load()


percentiles = {sim: native_percentile(sim) for sim in SIMS}

# Domain of the lat-lon datasets: points where the percentile is defined (NaN outside the CRCM6 domain)
domain = {}
for sim in LATLON:
    ref = merged['total_total'][sim]
    p = percentiles[sim].sel(quantile=QUANTILE, drop=True)
    assert np.array_equal(p['latitude'].values, ref['latitude'].values), sim
    assert np.array_equal(p['longitude'].values, ref['longitude'].values), sim
    domain[sim] = p.notnull().drop_vars([c for c in p.coords if c not in p.dims])
    print(f'{sim}: {int(domain[sim].sum())} grid points in the domain')

ERA5: 137086 grid points in the domain
IMERG: 854778 grid points in the domain


## Grids and regridders

In [3]:
def latlon_grid(lat, lon):
    # Regular lat-lon grid with the longitudes in 0..360 and sorted (contiguous over the domain), cell bounds
    # halfway between the centres. Also returns the order of the original columns on the grid.
    order = np.argsort(lon % 360, kind='stable')
    lon = (lon[order] % 360).astype(np.float64)
    lat = lat.astype(np.float64)

    def bounds(c):
        mid = (c[1:] + c[:-1]) / 2
        return np.concatenate([[2 * c[0] - mid[0]], mid, [2 * c[-1] - mid[-1]]])

    lon2d, lat2d = np.meshgrid(lon, lat)
    lon_b, lat_b = np.meshgrid(bounds(lon), np.clip(bounds(lat), -90, 90))
    grid = xr.Dataset(coords={'lat': (('latitude', 'longitude'), lat2d), 'lon': (('latitude', 'longitude'), lon2d),
                              'lat_b': (('latitude_b', 'longitude_b'), lat_b),
                              'lon_b': (('latitude_b', 'longitude_b'), lon_b)})
    return grid, order


def corners(c):
    # Cell corners of a 2-D grid as in regrid_over_ERA5_grid.ipynb: mean of the four surrounding centres, linearly
    # extrapolated at the edges
    b = np.zeros((c.shape[0] + 1, c.shape[1] + 1))
    b[1:-1, 1:-1] = (c[:-1, :-1] + c[1:, :-1] + c[:-1, 1:] + c[1:, 1:]) / 4
    b[0, 1:-1] = 2 * b[1, 1:-1] - b[2, 1:-1]
    b[-1, 1:-1] = 2 * b[-2, 1:-1] - b[-3, 1:-1]
    b[:, 0] = 2 * b[:, 1] - b[:, 2]
    b[:, -1] = 2 * b[:, -2] - b[:, -3]
    return b


ref_crcm6 = merged['total_total']['UBB']
lat_c = ref_crcm6['lat'].values.astype(np.float64)
lon_c = ref_crcm6['lon'].values.astype(np.float64) % 360   # the domain does not reach 0E: contiguous in 0..360
assert lon_c.min() > 90 and np.abs(np.diff(lon_c, axis=1)).max() < 10, 'CRCM6 longitudes not contiguous in 0..360'
grids = {'UBB': xr.Dataset(coords={'lat': (('rlat', 'rlon'), lat_c), 'lon': (('rlat', 'rlon'), lon_c),
                                   'lat_b': (('rlat_b', 'rlon_b'), corners(lat_c)),
                                   'lon_b': (('rlat_b', 'rlon_b'), corners(lon_c))})}
order = {}
for sim in LATLON:
    ref = merged['total_total'][sim]
    grids[sim], order[sim] = latlon_grid(ref['latitude'].values, ref['longitude'].values)
    print(f'{sim}: longitudes {grids[sim].lon.values.min():.2f} to {grids[sim].lon.values.max():.2f} E, '
          f'max step {np.diff(grids[sim].lon.values[0]).max():.3f}')

regridders = {}


def regridder(src, method):
    if (src, method) not in regridders:
        print(f'Regridder {src} -> ERA5 ({method})', flush=True)
        regridders[src, method] = xe.Regridder(grids[src], grids['ERA5'], method=method, unmapped_to_nan=True)
    return regridders[src, method]


def to_source_grid(da, sim):
    # Field on the grid of the regridders: NaN outside the domain and sorted longitudes for IMERG
    if sim in LATLON:
        return da.where(domain[sim]).isel(longitude=order[sim]).drop_vars(['latitude', 'longitude'])
    return da.drop_vars([c for c in ('lat', 'lon', 'rlat', 'rlon', 'rotated_pole') if c in da.coords])


def on_era5(da):
    # Coordinates of the ERA5 grid as in the merged contributions (original order of the longitudes), NaN outside
    # the domain
    ref = merged['total_total']['ERA5']
    da = da.drop_vars(['lat', 'lon'], errors='ignore').isel(longitude=np.argsort(order['ERA5']))
    da = da.assign_coords(latitude=ref['latitude'].values, longitude=ref['longitude'].values)
    return da.where(domain['ERA5'])


def to_era5(da, sim, method):
    # Field of sim (native grid) on the ERA5 grid, NaN outside the domain
    if sim == 'ERA5':
        return da.where(domain['ERA5'])
    out = regridder(sim, method)(to_source_grid(da, sim), skipna=True, na_thres=NA_THRES, keep_attrs=True)
    return on_era5(out)

ERA5: longitudes 174.25 to 357.00 E, max step 0.250
IMERG: longitudes 174.25 to 357.05 E, max step 0.100


## Contributions (all the keys of the merged file)

In [4]:
out = {}
for key, by_sim in merged.items():
    method = 'conservative' if key.startswith('total') else 'nearest_s2d'
    out[key] = {sim: to_era5(by_sim[sim], sim, method) for sim in SIMS}
path = f'{BASE}/ALL/CONTRIB/contribution_ETCs_1000hPa_{PERIOD}{TAG}_regridded_on_ERA5.pkl'
with open(f'{path}.tmp', 'wb') as f:
    pickle.dump(out, f)
os.replace(f'{path}.tmp', path)
print(f'Saved {path}', flush=True)
print('  mean of total_ext_EETCs_total over the domain (p99, mm over the period): ' + ', '.join(
    f'{sim} {float(da.sel(variable="pr", quantile=QUANTILE).mean()):.0f}'
    for sim, da in out['total_ext_EETCs_total'].items()))
del out

Regridder UBB -> ERA5 (conservative)


Regridder IMERG -> ERA5 (conservative)


Regridder UBB -> ERA5 (nearest_s2d)


Regridder IMERG -> ERA5 (nearest_s2d)


Saved /home/vdemeyer/projects/rrg-gachon/vdemeyer/ALL/CONTRIB/contribution_ETCs_1000hPa_2001-2014_noTC_regridded_on_ERA5.pkl


  mean of total_ext_EETCs_total over the domain (p99, mm over the period): ERA5 682, UBB 894, IMERG 803


## Percentiles (bilinear)

In [5]:
regridded_percentiles = {}
for sim in ['UBB', 'IMERG']:
    p = to_era5(percentiles[sim], sim, 'bilinear')
    regridded_percentiles[sim] = p
    path = f'{BASE}/{sim}/PR_Percentile/pr_{sim.lower()}_percentile_{PERIOD}_regridded_on_ERA5.nc'
    p.to_dataset(name='pr').to_netcdf(f'{path}.tmp')
    os.replace(f'{path}.tmp', path)
    print(f'Saved {path}; mean p99 {float(p.sel(quantile=QUANTILE).mean()):.3f} mm/h '
          f'(native {float(percentiles[sim].sel(quantile=QUANTILE).mean()):.3f})', flush=True)

Regridder UBB -> ERA5 (bilinear)


Saved /home/vdemeyer/projects/rrg-gachon/vdemeyer/UBB/PR_Percentile/pr_ubb_percentile_2001-2014_regridded_on_ERA5.nc; mean p99 1.702 mm/h (native 2.120)


Regridder IMERG -> ERA5 (bilinear)


Saved /home/vdemeyer/projects/rrg-gachon/vdemeyer/IMERG/PR_Percentile/pr_imerg_percentile_2001-2014_regridded_on_ERA5.nc; mean p99 1.706 mm/h (native 1.705)


## Significance of the differences with IMERG

In [6]:
def yearly(sim, kind):
    # Seasonal sum ('total') or count ('count') of the extreme precipitation within 1000 km of a cyclone for each
    # year of PERIOD, dims (season, year, y, x), native grid, NaN outside the domain
    first, last = map(int, PERIOD.split('-'))
    files = [f'{BASE}/{sim}/PR/1000km_storm/{kind}_pr_ext_{sim.lower()}_1000hPa_1000km_extreme_storm_'
             f'{season}_{year}_{PERIOD}{TAG}.nc' for year in range(first, last + 1) for season in SEASONS]
    with xr.open_mfdataset(files) as ds:
        da = ds['pr'].sel(quantile=QUANTILE, drop=True).load()
    if sim in LATLON:
        ref = merged['total_total'][sim]
        assert np.array_equal(da['latitude'].values, ref['latitude'].values), sim
        assert np.array_equal(da['longitude'].values, ref['longitude'].values), sim
    return da.sel(season=SEASONS).transpose('season', 'year', ...)


def yearly_on_era5(sim):
    # Yearly totals and counts on the ERA5 grid, dims (season, year, y, x)
    total = to_era5(yearly(sim, 'total'), sim, 'conservative')
    count = to_era5(yearly(sim, 'count'), sim, 'nearest_s2d')
    return [da.transpose('season', 'year', *DIMS) for da in (total, count)]


def era5_grid_percentile(sim):
    p = percentiles[sim].where(domain[sim]) if sim == 'ERA5' else regridded_percentiles[sim]
    return p.sel(quantile=QUANTILE, drop=True).transpose(*DIMS).values


print(f'Significance on the ERA5 grid (FDR {ALPHA_FDR})', flush=True)
ee = {}
for sim in TESTED + ['IMERG']:
    total, count = yearly_on_era5(sim)
    ee[sim] = with_annual(total - era5_grid_percentile(sim) * count)
    if sim == 'IMERG':
        # Check: the yearly files add up to the regridded total of the merged file
        ref = to_era5(merged['total_ext_EETCs_total']['IMERG'].sel(variable='pr', quantile=QUANTILE, drop=True),
                      'IMERG', 'conservative').transpose(*DIMS)
        diff = np.abs(total.sum(['season', 'year']).values - ref.values)
        print(f'  Check IMERG: yearly files vs merged total, max relative difference '
              f'{np.nanmax(diff) / np.nanmax(ref.values):.1e}', flush=True)
masks = {}
for sim in TESTED:
    print(f'  {sim} - IMERG', flush=True)
    masks[sim] = test_maps(ee[sim], ee['IMERG'])
ds = xr.concat([masks[sim] for sim in TESTED], dim=xr.DataArray(TESTED, dims='sim', name='sim')).to_dataset(name='EEcum_pr')
ds.attrs = {'description': '1 where the difference with IMERG is significant with respect to interannual variability',
            'test': "two-sided Welch's t-test on the yearly (or seasonal) values, " + PERIOD,
            'multiple_testing': f'false discovery rate controlled at {ALPHA_FDR} over each map (Wilks, 2016)',
            'grid': 'ERA5 grid (POSTPROCESSING/regrid_IMERG_evaluation.ipynb)'}
path = f'{BASE}/ALL/SIGNIFICANCE/significance_vs_IMERG_{PERIOD}_on_ERA5{TAG}.nc'
ds.to_netcdf(f'{path}.tmp', encoding={'EEcum_pr': {'zlib': True, 'dtype': 'int8'}})
os.replace(f'{path}.tmp', path)
print(f'Saved {path}', flush=True)

Significance on the ERA5 grid (FDR 0.1)


  Check IMERG: yearly files vs merged total, max relative difference 6.2e-08


  ERA5 - IMERG


    ALL:  14.3% of 128212 points significant


    DJF:   1.3% of 120431 points significant


    MAM:   0.6% of 118588 points significant


... (5 outputs left out)

    MAM:   0.0% of 120542 points significant


    JJA:   0.0% of 117593 points significant


    SON:   0.0% of 131038 points significant


Saved /home/vdemeyer/projects/rrg-gachon/vdemeyer/ALL/SIGNIFICANCE/significance_vs_IMERG_2001-2014_on_ERA5_noTC.nc
